In [ ]:
#!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#!pip install --no-cache-dir "trl==0.20.0" xformers vllm gguf pybase64 cbor2

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

In [ ]:
# Ensures your T4/L4 GPU is active
!nvidia-smi

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
SYSTEM_PROMPT = "Count letters. Use <reasoning> and <answer> tags."

In [ ]:
WORDS = ["APPLE", "BANANA", "CHERRY", "DOG", "ELEPHANT"]
data_dict = {"word": WORDS, "length": [len(w) for w in WORDS]}
ds = Dataset.from_dict(data_dict)

In [ ]:
def format_ds(example):
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                   {"role": "user", "content": f"Word: {example['word']}"}],
        "answer": example['length']
    }
dataset = ds.map(format_ds)

In [ ]:
def reward_func(completions, answer, **kwargs):
    # Simple logic: +2.0 for correct digit in answer
    return [2.0 if str(a) in c else 0.0 for c, a in zip(completions, answer)]

In [ ]:
from trl import GRPOConfig
training_args = GRPOConfig(
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    num_generations = 4,
    max_steps = 10,
    bf16 = False,
    generation_batch_size = 4,
)

In [ ]:
from trl import GRPOTrainer
trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_func],
    args = training_args,
    train_dataset = dataset,
)

In [ ]:
trainer.train()

In [ ]:
training_args.max_steps = 100
trainer.train()

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
messages = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Word: STRAWBERRY"}
], tokenize = True, add_generation_prompt = True,
return_tensors="pt", return_dict=True)

from vllm import SamplingParams
sampling_params = SamplingParams(temperature = 0.8, top_p = 0.95, max_tokens = 1024)

# Extract parameters from sampling_params object
output = model.fast_generate(
    **messages.to('cuda'),
    max_new_tokens = sampling_params.max_tokens,
    temperature = sampling_params.temperature,
    top_p = sampling_params.top_p,
)
print(output)

In [ ]:
decoded_output = tokenizer.decode(output[0])
print(decoded_output)

In [ ]:
test_prompt = "What is the capital of Japan?"
# Run the same generation code as above to verify 'Tokyo'

In [ ]:
from vllm import SamplingParams

# Test a word the model hasn't seen in the small training set
test_word = "STRAWBERRY"
prompt = [{"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": f"Word: {test_word}"}]

messages = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=1024)

# Generate the response
output = model.fast_generate(
    **messages.to('cuda'),
    max_new_tokens=sampling_params.max_tokens,
    temperature=sampling_params.temperature,
    top_p=sampling_params.top_p,
)
print(f"--- Model Output for {test_word} ---")
print(tokenizer.decode(output[0]))

In [ ]:
# Save the fine-tuned weights and tokenizer
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
# import shutil
# import os

# shutil.make_archive("submission", 'zip', base_dir=".")

# print("Archive 'submission.zip' created.")

In [ ]:
# import os
# from google.colab import files
# if os.path.exists("submission.zip"):
#     files.download("submission.zip")